<a href="https://colab.research.google.com/github/farrelrassya/IntroductionMachineLearningwithpython/blob/main/3_Chapter_3_Unsupervised_Learning_and_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised Learning (Pembelajaran Tanpa Pengawasan) & Preprocessing Data

Notebook ini menyajikan implementasi praktis dan penjelasan konseptual mengenai pembersihan awal data (*Preprocessing & Scaling*) serta algoritma *Unsupervised Learning* (seperti PCA, t-SNE, K-Means, Hierarchical Clustering, dan DBSCAN) berdasarkan Bab 3 dari buku *"Introduction to Machine Learning with Python"* (Müller & Guido).

## Bagian 1: Preprocessing & Scaling Data

Algoritma machine learning (seperti SVM atau Neural Networks) sangat sensitif terhadap skala fitur data input. Oleh karena itu, kita membagi pemrosesan awal menjadi 4 jenis penskalaan:
- **StandardScaler**: Menstandarkan nilai agar memiliki rata-rata (*mean*) = 0 dan variansi = 1.
- **MinMaxScaler**: Memetakan nilai fitur ke rentang batas bawah 0 hingga batas atas 1.
- **RobustScaler**: Mirip StandardScaler tetapi menggunakan nilai median dan kuartil sehingga kebal terhadap data pencilan (*outliers*).
- **Normalizer**: Melakukan penskalaan vektor sehingga panjang baris sampel bernilai tepat 1 (sering dipakai dalam pengolahan teks).

In [ ]:
# Setup environment & install libraries
!pip install mglearn seaborn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mglearn

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer, load_digits, make_moons, make_blobs
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.svm import SVC
from scipy.cluster.hierarchy import dendrogram, linkage

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Semua pustaka utama untuk Unsupervised Learning berhasil dimuat.")

In [ ]:
# Membuat dataset sintetis sederhana untuk visualisasi skaler
X_sintetis, y_sintetis = make_blobs(n_samples=50, centers=2, random_state=4, cluster_std=1)
X_latih, X_uji, y_latih, y_uji = train_test_split(X_sintetis, y_sintetis, random_state=5, test_size=0.25)

# Visualisasi perbandingan efek penskalaan
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# 1. Plot Data Asli sebelum penskalaan
axes[0].scatter(X_latih[:, 0], X_latih[:, 1], c=y_latih, cmap=mglearn.cm2, label="Latih")
axes[0].scatter(X_uji[:, 0], X_uji[:, 1], c=y_uji, cmap=mglearn.cm2, marker='^', label="Uji")
axes[0].set_title("Data Asli")
axes[0].legend(loc="upper left")

daftar_skaler = [StandardScaler(), MinMaxScaler(), RobustScaler(), Normalizer()]
nama_skaler = ["StandardScaler", "MinMaxScaler", "RobustScaler", "Normalizer"]

# Melakukan fitting dan visualisasi untuk masing-masing skaler
for ax, skaler_uji, nama in zip(axes[1:], daftar_skaler, nama_skaler):
    X_latih_skala = skaler_uji.fit_transform(X_latih)
    X_uji_skala = skaler_uji.transform(X_uji)
    ax.scatter(X_latih_skala[:, 0], X_latih_skala[:, 1], c=y_latih, cmap=mglearn.cm2)
    ax.scatter(X_uji_skala[:, 0], X_uji_skala[:, 1], c=y_uji, cmap=mglearn.cm2, marker='^')
    ax.set_title(nama)

plt.suptitle("Perbandingan Efek Berbagai Metode Penskalaan (Scalers)", y=1.05, fontsize=16)
plt.show()

In [ ]:
# Demonstrasi kesalahan penskalaan (fit ulang pada test set) vs penskalaan yang benar (konsisten fit data latih)
X_kebocoran, _ = make_blobs(n_samples=50, centers=5, random_state=4, cluster_std=2)
X_latih_k, X_uji_k = train_test_split(X_kebocoran, random_state=5, test_size=0.1)

# Penskalaan Konsisten (Benar)
skaler_konsisten = MinMaxScaler()
X_train_benar = skaler_konsisten.fit_transform(X_latih_k)
X_uji_benar = skaler_konsisten.transform(X_uji_k) # hanya transformasi uji berdasarkan latih

# Penskalaan Terpisah (Salah - Mengakibatkan Data Leakage)
skaler_independen = MinMaxScaler()
X_train_salah = skaler_independen.fit_transform(X_latih_k)
X_uji_salah = skaler_independen.fit_transform(X_uji_k) # Salah! memanggil kembali fit pada data uji

# Visualisasi pergeseran titik koordinat
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].scatter(X_latih_k[:, 0], X_latih_k[:, 1], label="Latih", alpha=0.7)
axes[0].scatter(X_uji_k[:, 0], X_uji_k[:, 1], label="Uji", marker='^', s=100)
axes[0].set_title("Data Asli")
axes[0].legend()

axes[1].scatter(X_train_benar[:, 0], X_train_benar[:, 1], label="Latih", alpha=0.7)
axes[1].scatter(X_uji_benar[:, 0], X_uji_benar[:, 1], label="Uji", marker='^', s=100)
axes[1].set_title("Penskalaan yang Benar (Konsisten)")

axes[2].scatter(X_train_salah[:, 0], X_train_salah[:, 1], label="Latih", alpha=0.7)
axes[2].scatter(X_uji_salah[:, 0], X_uji_salah[:, 1], label="Uji", marker='^', s=100)
axes[2].set_title("Penskalaan yang Salah (Fit Ulang Uji)")

plt.suptitle("Dampak Negatif Kebocoran Informasi (Data Leakage) Akibat Salah Scaling", y=1.02, fontsize=14)
plt.show()

In [ ]:
# Pengaruh Penskalaan terhadap Performa SVM (Breast Cancer Dataset)
kanker = load_breast_cancer()
X_latih_kan, X_uji_kan, y_latih_kan, y_uji_kan = train_test_split(
    kanker.data, kanker.target, random_state=0
)

# SVM Tanpa Scaling
svm_tanpa_skala = SVC(C=100).fit(X_latih_kan, y_latih_kan)
print(f"Akurasi Model SVM Tanpa Penskalaan: {svm_tanpa_skala.score(X_uji_kan, y_uji_kan):.4f}")

# SVM dengan MinMaxScaler
skaler_kanker = MinMaxScaler()
X_latih_scaled = skaler_kanker.fit_transform(X_latih_kan)
X_uji_scaled = skaler_kanker.transform(X_uji_kan)

svm_dengan_skala = SVC(C=100).fit(X_latih_scaled, y_latih_kan)
print(f"Akurasi Model SVM dengan MinMaxScaler  : {svm_dengan_skala.score(X_uji_scaled, y_uji_kan):.4f}")

## Bagian 2: Reduksi Dimensi & Ekstraksi Fitur (PCA vs t-SNE)

- **PCA (Principal Component Analysis)**: Mereduksi jumlah fitur dengan memproyeksikan data ke arah variansi maksimum (membentuk komponen utama/PC yang saling ortogonal).
- **t-SNE (t-Distributed Stochastic Neighbor Embedding)**: Metode reduksi dimensi non-linier (*manifold learning*) yang memetakan titik data berdekatan di dimensi tinggi agar tetap berdekatan di dimensi rendah. Sangat baik untuk visualisasi data kompleks.

In [ ]:
# Mereduksi dimensi dataset Kanker dari 30 fitur menjadi 2 fitur (PC1 & PC2)
skaler_std = StandardScaler()
X_kanker_skala = skaler_std.fit_transform(kanker.data)

# Menerapkan PCA
pca_kanker = PCA(n_components=2)
X_kanker_pca = pca_kanker.fit_transform(X_kanker_skala)

print(f"Dimensi Kanker Sebelum Reduksi: {X_kanker_skala.shape}")
print(f"Dimensi Kanker Setelah Reduksi: {X_kanker_pca.shape}")

# Visualisasi sebaran kelas tumor
plt.figure(figsize=(8, 6))
mglearn.discrete_scatter(X_kanker_pca[:, 0], X_kanker_pca[:, 1], kanker.target)
plt.legend(kanker.target_names, loc="best")
plt.xlabel("Komponen Utama Pertama (PC1)")
plt.ylabel("Komponen Utama Kedua (PC2)")
plt.title("Sebaran Data 2D Komponen Utama (PCA) Kanker Payudara")
plt.show()

### Visualisasi Kinerja Reduksi Dimensi: PCA vs t-SNE pada Dataset Digits

Kita menggunakan dataset citra tulisan tangan angka *Digits* (64 dimensi) untuk membandingkan kemampuan PCA dan t-SNE dalam memisahkan 10 kelas target angka dalam ruang visual 2D.

In [ ]:
# Memuat dataset Digits
dataset_digits = load_digits()

# 1. Reduksi 2D dengan PCA
pca_dig = PCA(n_components=2)
X_digits_pca = pca_dig.fit_transform(dataset_digits.data)

# 2. Reduksi 2D dengan t-SNE
tsne_dig = TSNE(random_state=42)
X_digits_tsne = tsne_dig.fit_transform(dataset_digits.data)

# Visualisasi komparatif
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
warna_kelas = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
               "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

# Plot Proyeksi PCA
for i in range(10):
    axes[0].scatter(X_digits_pca[dataset_digits.target == i, 0], X_digits_pca[dataset_digits.target == i, 1],
                    color=warna_kelas[i], label=str(i), alpha=0.6)
axes[0].set_xlabel("Komponen Utama 0 (PC1)")
axes[0].set_ylabel("Komponen Utama 1 (PC2)")
axes[0].set_title("Proyeksi Visualisasi 2D dengan PCA")

# Plot Proyeksi t-SNE
for i in range(10):
    axes[1].scatter(X_digits_tsne[dataset_digits.target == i, 0], X_digits_tsne[dataset_digits.target == i, 1],
                    color=warna_kelas[i], label=str(i), alpha=0.6)
axes[1].set_xlabel("Fitur t-SNE 0")
axes[1].set_ylabel("Fitur t-SNE 1")
axes[1].set_title("Proyeksi Visualisasi 2D dengan t-SNE")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.suptitle("Perbandingan Visual Proyeksi Dimensi: PCA vs t-SNE", y=0.98, fontsize=16)
plt.show()

## Bagian 3: Algoritma Pengelompokan (Clustering)

Pengelompokan digunakan untuk membagi dataset ke dalam kelompok-kelompok homogen tanpa adanya label kelas penunjuk.
- **K-Means**: Mengelompokkan data berdasarkan jarak Euclidean terdekat ke titik pusat (*centroid*).
- **Agglomerative (Hierarchical)**: Menggabungkan titik data terdekat secara hierarki bertahap hingga menyisakan klaster utama (visualisasinya disebut *dendrogram*).
- **DBSCAN**: Pengelompokan berbasis kepadatan densitas data lokal, kebal terhadap outliers dan mampu mengidentifikasi klaster dengan struktur bentuk non-bulat/rumit.

In [ ]:
# Melatih K-Means pada dataset sintetis sebaran blobs
X_blobs, _ = make_blobs(random_state=1, n_samples=200)

# Melatih K-Means dengan K = 3 klaster
kmeans = KMeans(n_clusters=3, random_state=0)
labels_kmeans = kmeans.fit_predict(X_blobs)

# Visualisasi pengelompokan K-Means dan centroidnya
plt.figure(figsize=(7, 5))
mglearn.discrete_scatter(X_blobs[:, 0], X_blobs[:, 1], labels_kmeans, markers='o')
mglearn.discrete_scatter(
    kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], [0, 1, 2],
    markers='^', markeredgewidth=4, s=150
)
plt.title("K-Means Clustering dengan Visualisasi Titik Pusat Centroid (Segitiga)")
plt.show()

In [ ]:
# Kegagalan K-Means pada dataset non-sferis (make_moons)
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=0)
kmeans_moons = KMeans(n_clusters=2, random_state=0).fit(X_moons)

plt.figure(figsize=(7, 5))
mglearn.discrete_scatter(X_moons[:, 0], X_moons[:, 1], kmeans_moons.labels_)
plt.title("Kegagalan K-Means dalam Mengelompokkan Struktur Klaster Melengkung")
plt.show()

### Agglomerative Clustering & Dendrogram

Kita memvisualisasikan matriks kedekatan jarak pengelompokan hierarki bertahap menggunakan diagram pohon (*dendrogram*).

In [ ]:
# Membuat representasi Dendrogram
X_hierarki, _ = make_blobs(random_state=0, n_samples=12)

# Menghitung linkage matrix menggunakan metode Ward link
matriks_jarak_ward = linkage(X_hierarki, method='ward')

plt.figure(figsize=(10, 6))
dendrogram(matriks_jarak_ward)
plt.title("Dendrogram Pengelompokan Hierarkis (Agglomerative)")
plt.xlabel("Indeks Sampel Titik Data")
plt.ylabel("Jarak Relatif Antar Klaster")
plt.show()

### Pengelompokan Berbasis Kepadatan (DBSCAN)

DBSCAN mampu memetakan kelompok dengan bentuk sirkuler rumit secara sempurna karena mendeteksi kerapatan densitas lokal titik data di sekitarnya.

In [ ]:
# Melatih model DBSCAN pada dataset make_moons
dbscan_moons = DBSCAN(eps=0.2, min_samples=5)
labels_dbscan = dbscan_moons.fit_predict(X_moons)

plt.figure(figsize=(7, 5))
mglearn.discrete_scatter(X_moons[:, 0], X_moons[:, 1], labels_dbscan)
plt.title("Keberhasilan DBSCAN dalam Memetakan Bentuk Klaster Kompleks")
plt.show()

## Bagian 4: Evaluasi & Perbandingan Clustering

Mengukur performa klastering menggunakan metrik:
- **Adjusted Rand Index (ARI)**: Membandingkan label klaster dengan label asli (*ground truth*). Rentang nilai -1 hingga 1 (skor 1 melambangkan kesamaan sempurna).
- **Silhouette Score**: Mengukur kerapatan dalam klaster dan kerenggangan antar klaster (tanpa memerlukan ground truth).

In [ ]:
# Membandingkan K-Means, Agglomerative, dan DBSCAN pada dataset Moons
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

daftar_algoritma = [
    KMeans(n_clusters=2, random_state=0),
    AgglomerativeClustering(n_clusters=2),
    DBSCAN(eps=0.2, min_samples=5)
]

for ax, algoritma in zip(axes, daftar_algoritma):
    prediksi_klaster = algoritma.fit_predict(X_moons)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=prediksi_klaster, cmap=mglearn.cm3, s=60)
    
    # Menghitung skor metrik
    skor_ari = adjusted_rand_score(y_moons, prediksi_klaster)
    skor_silhouette = silhouette_score(X_moons, prediksi_klaster)
    
    ax.set_title(f"{algoritma.__class__.__name__}\nARI: {skor_ari:.3f} | Silhouette: {skor_silhouette:.3f}")

plt.suptitle("Analisis Perbandingan Algoritma Pengelompokan (Clustering)", y=1.05, fontsize=16)
plt.show()